In [1]:
#BSWC - Building Systems with ChatGPT API

#Language Models, the Chat Format and Tokens

In [2]:
#1.SETTING UP THE OPENAI CLIENT

import os
# import → Keyword used to bring a library into the notebook
# os → Built-in library for interacting with the operating system (e.g., reading environment variables)

from dotenv import load_dotenv
# from → Keyword to import from a specific library
# dotenv → Library (python-dotenv) for reading key-value settings from a .env file
# import → Keyword used to bring in the specified item
# load_dotenv → Function that loads variables from a .env file into the environment

from openai import OpenAI
# from → Keyword to import from a specific library
# openai → Library used to connect to OpenAI's models
# import → Keyword used to bring in the specified item
# OpenAI → Class used to create a client that sends requests to OpenAI's API

load_dotenv(dotenv_path=".env")
# load_dotenv → Function that loads variables from a .env file into the environment
# (dotenv_path=".env") → Argument specifying the path/name of the file to read

openai_api_key = os.getenv("OPENAI_API_KEY")
# openai_api_key → Variable name storing the result
# = → Assignment operator
# os → Built-in operating system library
# .getenv → Function that reads the value of an environment variable
# ("OPENAI_API_KEY") → Argument specifying the name of the variable to read (your secret API key)

client = OpenAI()
# client → Variable name storing the result
# = → Assignment operator
# OpenAI() → Creates the API client object (automatically uses the API key from the environment)

# This cell loads the API key from a .env file and creates the OpenAI client object used by later cells to send requests to the model.

In [3]:
#2.FUNCTION TO GET A COMPLETION FROM THE MODEL

def get_completion(prompt, model="gpt-5.6-luna"):
# def → Keyword used to define a function
# get_completion → Name given to the function
# (prompt, model="gpt-5.6-luna") → Parameters: the text to send, and the model to use (defaults to gpt-5.6-luna if not given)
# : → Marks the start of the function body

    messages = [{"role": "user", "content": prompt}]
    # messages → Variable name storing the result
    # = → Assignment operator
    # [ ] → List brackets holding the conversation history
    # {"role": "user" → Key-value pair marking this message as coming from the user
    # "content": prompt} → Key-value pair holding the text passed into the function

    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=1,
    )
    # response → Variable name storing the result
    # = → Assignment operator
    # client → The OpenAI client object created earlier
    # .chat.completions → Attribute giving access to the Chat Completions API
    # .create → Function that sends a request to the model and gets a reply
    # ( → Opens the function's arguments
    # model=model → Argument passing the model name given to the function
    # messages=messages, → Argument passing the single-message conversation
    # temperature=1, → Argument setting randomness to 1, the default/only supported value for gpt-5.6-luna
    # ) → Closes the function's arguments

    return response.choices[0].message.content
    # return → Keyword that sends a value back from the function
    # response → The reply object returned by the model
    # .choices → Attribute holding a list of possible responses (usually just one)
    # [0] → Indexing operator selecting the first (and only) response
    # .message → Attribute holding the message object of that response
    # .content → Attribute holding the actual answer text

# This function sends a single prompt to the specified model (defaulting to gpt-5.6-luna) with temperature=1, 
# and returns just the text of the model's reply.

In [4]:
#3.CALLING THE get_completion FUNCTION WITH A SAMPLE QUESTION

response = get_completion("What is the capital of France?")
# response → Variable name storing the result
# = → Assignment operator
# get_completion → The function defined earlier that sends a prompt and returns the model's reply
# ("What is the capital of France?") → Argument passing the question as the prompt text

# This cell calls get_completion with a sample question and stores the model's answer in response (not yet printed — use print(response) 
# in the next cell to see it).

In [5]:
#4.PRINTING THE MODEL'S RESPONSE

print(response)
# print → Built-in function that displays output
# (response) → Argument specifying the variable to display — the answer stored earlier from get_completion("What is the capital of France?")

# This cell prints the stored response, showing the model's answer to "What is the capital of France?".

The capital of France is **Paris**.


In [6]:
#5.TESTING THE MODEL WITH A LETTER-REVERSAL TASK

response = get_completion("Take the letters in lollipop \
and reverse them")
# response → Variable name storing the result
# = → Assignment operator
# get_completion → The function defined earlier that sends a prompt and returns the model's reply
# ("Take the letters in lollipop \ and reverse them") → Argument passing the instruction as the prompt text
# \ → Line continuation character: lets a single string be written across two lines in the code without inserting a line break into the actual text

print(response)
# print → Built-in function that displays output
# (response) → Argument specifying what to display — the model's attempt at reversing "lollipop"

# This cell tests the model on a character-level task: reversing the letters of the word "lollipop", then prints the result.

popillol


In [7]:
#6.TESTING LETTER-REVERSAL WITH HYPHEN-SEPARATED LETTERS

response = get_completion("""Take the letters in \
l-o-l-l-i-p-o-p and reverse them""")
# response → Variable name storing the result
# = → Assignment operator
# get_completion → The function defined earlier that sends a prompt and returns the model's reply
# (""" ... """) → Triple-quoted string, letting the prompt text span multiple lines in the code
# "Take the letters in l-o-l-l-i-p-o-p and reverse them" → The actual prompt text, with each letter of "lollipop" separated by hyphens
# \ → Line continuation character: joins the two code lines into one continuous string with no line break in the text

print(response)
# print → Built-in function that displays output
# (response) → Argument specifying what to display — the model's attempt at reversing the hyphen-separated letters

# This cell retests letter-reversal, but this time spells "lollipop" with hyphens between each letter (l-o-l-l-i-p-o-p), then prints the result. 
# Hyphens often help the model succeed at character-level tasks, because each letter becomes its own token instead of being grouped into one word token.

p-o-p-i-l-l-o-l


In [10]:
#7.FUNCTION TO GET A COMPLETION WITH CUSTOM TEMPERATURE AND MAX TOKENS

def get_completion_from_messages(messages,
                                  model="gpt-5.6-luna",
                                  temperature=1,
                                  max_tokens=500):
# def → Keyword used to define a function
# get_completion_from_messages → Name given to the function
# (messages, model="gpt-5.6-luna", temperature=1, max_tokens=500) → Parameters: the conversation list, and three settings with default values
# messages → Parameter holding the full list of conversation messages (no default, must be provided)
# model="gpt-5.6-luna" → Parameter setting which model to use, defaulting to gpt-5.6-luna
# temperature=1 → Parameter controlling randomness, defaulting to 1 (gpt-5.6-luna only supports the default)
# max_tokens=500 → Parameter limiting the length of the reply, defaulting to 500 tokens (passed into max_completion_tokens below)
# : → Marks the start of the function body

    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature, # this is the degree of randomness of the model's output
        max_completion_tokens=max_tokens, # the maximum number of tokens the model can output
    )
    # response → Variable name storing the result
    # = → Assignment operator
    # client → The OpenAI client object created earlier
    # .chat.completions → Attribute giving access to the Chat Completions API
    # .create → Function that sends a request to the model and gets a reply
    # ( → Opens the function's arguments
    # model=model → Argument passing the model name given to the function
    # messages=messages, → Argument passing the conversation list given to the function
    # temperature=temperature, → Argument passing the randomness setting given to the function
    # # this is the degree of randomness of the model's output → Comment explaining what temperature controls
    # max_completion_tokens=max_tokens, → Argument passing the output length limit, using the parameter name required by gpt-5.6-luna
    # # the maximum number of tokens the model can output → Comment explaining what this setting controls
    # ) → Closes the function's arguments

    return response.choices[0].message.content
    # return → Keyword that sends a value back from the function
    # response → The reply object returned by the model
    # .choices → Attribute holding a list of possible responses (usually just one)
    # [0] → Indexing operator selecting the first (and only) response
    # .message → Attribute holding the message object of that response
    # .content → Attribute holding the actual answer text

# This function accepts a full messages list and lets you customize the model, temperature, and output length, returning just the model's reply text. 
# It uses max_completion_tokens instead of the older max_tokens, since gpt-5.6-luna requires the newer parameter name.

In [11]:
#8.USING THE MESSAGES FUNCTION WITH A SYSTEM PERSONA AND USER REQUEST

messages = [
{'role':'system',
 'content':"""You are an assistant who\
 responds in the style of Dr Seuss."""},
{'role':'user',
 'content':"""write me a very short poem\
 about a happy carrot"""},
]
# messages → Variable name storing the result
# = → Assignment operator
# [ ] → List brackets holding the full conversation history
# {'role':'system' → Key-value pair marking this message as the system instruction
# 'content':"""You are an assistant who responds in the style of Dr Seuss."""} → Key-value pair setting the assistant's fixed persona (writing style)
# \ → Line continuation character: joins the two code lines into one continuous string with no line break in the text
# {'role':'user' → Key-value pair marking this message as coming from the user
# 'content':"""write me a very short poem about a happy carrot"""} → Key-value pair holding the user's actual request

response = get_completion_from_messages(messages, temperature=1)
# response → Variable name storing the result
# = → Assignment operator
# get_completion_from_messages → The function defined earlier that sends a full message list and returns the model's reply
# (messages → Argument passing the conversation list defined above
# temperature=1) → Argument setting randomness to 1 (gpt-5.6-luna's only supported value)

print(response)
# print → Built-in function that displays output
# (response) → Argument specifying what to display — the Dr. Seuss-style poem about a happy carrot

# This cell sets the assistant's persona to write in the style of Dr. Seuss (system role), asks it to write a short poem about a happy carrot 
# (user role), sends both to get_completion_from_messages, and prints the resulting poem.

A happy carrot, bright and spry,  
Wiggled its leaves beneath the sky!


In [12]:
#9.CONTROLLING RESPONSE LENGTH WITH A SYSTEM INSTRUCTION

# length

messages = [
{'role':'system',
 'content':'All your responses must be \
one sentence long.'},
{'role':'user',
 'content':'write me a story about a happy carrot'},
]
# messages → Variable name storing the result
# = → Assignment operator
# [ ] → List brackets holding the full conversation history
# {'role':'system' → Key-value pair marking this message as the system instruction
# 'content':'All your responses must be one sentence long.'} → Key-value pair restricting the assistant's output length to a single sentence
# \ → Line continuation character: joins the two code lines into one continuous string with no line break in the text
# {'role':'user' → Key-value pair marking this message as coming from the user
# 'content':'write me a story about a happy carrot'} → Key-value pair holding the user's actual request

response = get_completion_from_messages(messages, temperature=1)
# response → Variable name storing the result
# = → Assignment operator
# get_completion_from_messages → The function defined earlier that sends a full message list and returns the model's reply
# (messages → Argument passing the conversation list defined above
# temperature=1) → Argument setting randomness to 1 (gpt-5.6-luna's only supported value)

print(response)
# print → Built-in function that displays output
# (response) → Argument specifying what to display — the one-sentence carrot story

# This cell tests controlling output length through a system instruction: it tells the assistant all responses must be one sentence long (system role),
# asks for a story about a happy carrot (user role), sends both to get_completion_from_messages, and prints the resulting one-sentence story.

Once upon a time, a bright orange carrot named Sunny danced in the garden each morning, made friends with the butterflies, and felt happiest whenever the gentle rain helped everyone grow.


In [14]:
#10.COMBINING PERSONA AND LENGTH INSTRUCTIONS

# combined

messages = [
{'role':'system',
 'content':"""You are an assistant who \
responds in the style of Dr Seuss. \
All your responses must be one sentence long."""},
{'role':'user',
 'content':"""write me a story about a happy carrot"""},
]
# messages → Variable name storing the result
# = → Assignment operator
# [ ] → List brackets holding the full conversation history
# {'role':'system' → Key-value pair marking this message as the system instruction
# 'content':"""You are an assistant who responds in the style of Dr Seuss. All your responses must be one sentence long."""} → Key-value pair combining 
# two rules: write in Dr. Seuss's style AND keep responses to one sentence

# \ → Line continuation character: joins multiple code lines into one continuous string with no line breaks in the text
# {'role':'user' → Key-value pair marking this message as coming from the user
# 'content':"""write me a story about a happy carrot"""} → Key-value pair holding the user's actual request

response = get_completion_from_messages(messages,
                                         temperature=1)
# response → Variable name storing the result
# = → Assignment operator
# get_completion_from_messages → The function defined earlier that sends a full message list and returns the model's reply
# (messages → Argument passing the conversation list defined above
# temperature=1) → Argument setting randomness to 1 (gpt-5.6-luna's only supported value)

print(response)
# print → Built-in function that displays output
# (response) → Argument specifying what to display — the one-sentence, Dr. Seuss-style carrot story

# This cell combines two system instructions at once: writing in the style of Dr. Seuss AND keeping every response to one sentence, 
# asks for a story about a happy carrot, sends both to get_completion_from_messages, and prints the resulting short, rhyming story.

In a sunshiny patch by a peppermint tree, a happy carrot named Carroline danced with the bees and grinned at the breeze.


In [15]:
#11.FUNCTION TO GET A COMPLETION PLUS TOKEN USAGE COUNTS

def get_completion_and_token_count(messages,
                                    model="gpt-5.6-luna",
                                    temperature=1,
                                    max_tokens=500):
# def → Keyword used to define a function
# get_completion_and_token_count → Name given to the function
# (messages, model="gpt-5.6-luna", temperature=1, max_tokens=500) → Parameters: the conversation list, and three settings with default values
# messages → Parameter holding the full list of conversation messages (no default, must be provided)
# model="gpt-5.6-luna" → Parameter setting which model to use, defaulting to gpt-5.6-luna
# temperature=1 → Parameter controlling randomness, defaulting to 1 (gpt-5.6-luna's only supported value)
# max_tokens=500 → Parameter limiting the length of the reply, defaulting to 500 tokens
# : → Marks the start of the function body

    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature,
        max_completion_tokens=max_tokens,
    )
    # response → Variable name storing the result
    # = → Assignment operator
    # client → The OpenAI client object created earlier
    # .chat.completions → Attribute giving access to the Chat Completions API
    # .create → Function that sends a request to the model and gets a reply
    # ( → Opens the function's arguments
    # model=model → Argument passing the model name given to the function
    # messages=messages, → Argument passing the conversation list given to the function
    # temperature=temperature, → Argument passing the randomness setting given to the function
    # max_completion_tokens=max_tokens, → Argument passing the output length limit, using the parameter name required by gpt-5.6-luna
    # ) → Closes the function's arguments

    content = response.choices[0].message.content
    # content → Variable name storing the result
    # = → Assignment operator
    # response → The reply object returned by the model
    # .choices → Attribute holding a list of possible responses (usually just one)
    # [0] → Indexing operator selecting the first (and only) response
    # .message → Attribute holding the message object of that response
    # .content → Attribute holding the actual answer text

    token_dict = {
        'prompt_tokens': response.usage.prompt_tokens,
        'completion_tokens': response.usage.completion_tokens,
        'total_tokens': response.usage.total_tokens,
    }
    # token_dict → Variable name storing the result
    # = → Assignment operator
    # { } → Curly braces creating a dictionary of key-value pairs
    # 'prompt_tokens': response.usage.prompt_tokens → Key holding the number of tokens used by the input/prompt
    # 'completion_tokens': response.usage.completion_tokens → Key holding the number of tokens used by the model's reply
    # 'total_tokens': response.usage.total_tokens → Key holding the combined total of prompt and completion tokens

    return content, token_dict
    # return → Keyword that sends values back from the function
    # content, token_dict → Two values returned together as a tuple: the reply text and the token usage dictionary

# This function sends messages to the model and returns both the reply text and a dictionary of token usage (prompt, completion, and total tokens), 
# useful for tracking cost and context usage.

In [17]:
#12.CALLING THE TOKEN-COUNTING FUNCTION WITH A DR. SEUSS PROMPT

messages = [
{'role':'system',
 'content':"""You are an assistant who responds\
 in the style of Dr Seuss."""},
{'role':'user',
 'content':"""write me a very short poem \
about a happy carrot"""},
]
# messages → Variable name storing the result
# = → Assignment operator
# [ ] → List brackets holding the full conversation history
# {'role':'system' → Key-value pair marking this message as the system instruction
# 'content':"""You are an assistant who responds in the style of Dr Seuss."""} → Key-value pair setting the assistant's fixed persona (writing style)
# \ → Line continuation character: joins the two code lines into one continuous string with no line break in the text
# {'role':'user' → Key-value pair marking this message as coming from the user
# 'content':"""write me a very short poem about a happy carrot"""} → Key-value pair holding the user's actual request

response, token_dict = get_completion_and_token_count(messages)
# response, token_dict → Two variables storing the two values returned by the function
# = → Assignment operator
# get_completion_and_token_count → The function defined earlier that returns both the reply text and a token usage dictionary
# (messages) → Argument passing the conversation list defined above

# This cell asks the assistant (persona: Dr. Seuss style) to write a short poem about a happy carrot, then calls get_completion_and_token_count to 
# get both the poem (response) and its token usage breakdown (token_dict) in one call.

In [18]:
#13.PRINTING THE POEM RESPONSE

print(response)
# print → Built-in function that displays output
# (response) → Argument specifying what to display — the Dr. Seuss-style poem stored from the previous cell

# This cell prints the poem text returned by get_completion_and_token_count in the previous cell.

I can’t write exactly in Dr. Seuss’s style, but here’s a bouncy, whimsical version:

A carrot danced in morning light,  
“Today,” it cheered, “feels just right!”


In [19]:
#14.PRINTING THE TOKEN USAGE DICTIONARY

print(token_dict)
# print → Built-in function that displays output
# (token_dict) → Argument specifying what to display — the dictionary holding prompt_tokens, completion_tokens, and total_tokens from the previous cell

# This cell prints the token usage breakdown returned by get_completion_and_token_count, showing how many tokens were used for the prompt, 
# the completion, and the total combined.

{'prompt_tokens': 34, 'completion_tokens': 107, 'total_tokens': 141}


Notes on using the OpenAI API outside of this classroom
To install the OpenAI Python library:

!pip install openai
The library needs to be configured with your account's secret key, which is available on the website.

You can either set it as the OPENAI_API_KEY environment variable before using the library:

!export OPENAI_API_KEY='sk-...'
Or, set openai.api_key to its value:

import openai
openai.api_key = "sk-..."

A note about the backslash

In the course, we are using a backslash \ to make the text fit on the screen without inserting newline '\n' characters.
GPT-3 isn't really affected whether you insert newline characters or not. But when working with LLMs in general, you may consider whether newline characters in your prompt may affect the model's performance.